# PPE Detector V2 — YOLO26n Baseline

This notebook establishes one reproducible baseline for four classes: `helmet`, `no-helmet`, `vest`, and `no-vest`. It answers a single question: **how well does the standard YOLO26n model learn Dataset V2 at 640 × 640?**

The notebook covers dataset verification, baseline training, checkpoint recovery, validation, learning curves, and model export. Architecture variants and camera-degradation experiments are intentionally excluded.

## 1. Experimental protocol

The baseline uses the official `yolo26n.pt` initialization, the complete Dataset V2 training split, an input size of 640 pixels, and seed 42. Model selection uses validation `mAP50-95`; the test split is not used in this notebook.

Outputs are stored separately under `/kaggle/working/ppe_baseline_study`. A completed epoch updates `last.pt`, while `best.pt` stores the checkpoint with the best validation fitness. When training continues in another Kaggle Version, the previous `best.pt` is staged beside `last.pt` so a weaker later epoch cannot erase the strongest earlier model.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

packages = []
try:
    if Version(version("ultralytics")) < Version("8.4.56"):
        packages.append("ultralytics>=8.4.56")
except PackageNotFoundError:
    packages.append("ultralytics>=8.4.56")
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Environment ready.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import random
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO
import ultralytics

DATASET_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
RESUME_WEIGHTS = None  # Path to the previous baseline last.pt when continuing another Kaggle Version.
PREVIOUS_BEST_WEIGHTS = None  # Optional path to best.pt; sibling best.pt is discovered automatically.
RUN_TRAINING = True
EPOCHS = 100
SEED = 42
SAVE_PERIOD = 5  # last.pt still updates every epoch; archive one epoch file every five epochs.
IMAGE_SIZE = 640
CLASS_NAMES = {0: "helmet", 1: "no-helmet", 2: "vest", 3: "no-vest"}
GPU_COUNT = torch.cuda.device_count()
USE_DUAL_GPU = GPU_COUNT >= 2
TRAIN_DEVICE = [0, 1] if USE_DUAL_GPU else (0 if GPU_COUNT == 1 else "cpu")
EVAL_DEVICE = 0 if GPU_COUNT >= 1 else "cpu"
BATCH = 32 if USE_DUAL_GPU else 16  # Global batch; dual GPU keeps 16 images per T4.
WORKERS = 2 if USE_DUAL_GPU else 4  # Two DDP processes share the Kaggle CPU workers.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Detected GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
print("Training device:", TRAIN_DEVICE, "| Global batch:", BATCH, "| Workers per process:", WORKERS)

In [ ]:
def is_dataset_root(path):
    val_name = "valid" if (path / "images" / "valid").exists() else "val"
    return (path / "images" / "train").exists() and (path / "images" / val_name).exists()

def extract_archive(mount):
    archives = sorted(mount.glob("*.rar"))
    if not archives:
        return None
    destination = Path("/kaggle/working/ppe_v2_data")
    destination.mkdir(parents=True, exist_ok=True)
    extractor = shutil.which("7z") or shutil.which("7zz")
    if not extractor:
        raise RuntimeError("Dataset_v2.rar was found but 7-Zip is unavailable. Upload the extracted dataset.")
    subprocess.check_call([extractor, "x", str(archives[0]), f"-o{destination}", "-y"])
    matches = [p for p in destination.rglob("*") if p.is_dir() and is_dataset_root(p)]
    return matches[0].resolve() if matches else None

def find_dataset_root():
    if DATASET_ROOT_OVERRIDE:
        root = Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
        if is_dataset_root(root):
            return root
    mounts = [
        Path("/kaggle/input/datasets/linhlaz225/helmet-nohelmet-vest-novest"),
        Path("/kaggle/input/helmet-nohelmet-vest-novest"),
    ]
    local_candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        local_candidates.extend([parent / "Dataset_v2", parent / "Dataset" / "Dataset_v2"])
    for candidate in [*(m / "Dataset_v2" for m in mounts), *mounts, *local_candidates]:
        if is_dataset_root(candidate):
            return candidate.resolve()
    for mount in mounts:
        if mount.exists():
            nested = [p for p in mount.rglob("*") if p.is_dir() and is_dataset_root(p)]
            if nested:
                return nested[0].resolve()
            extracted = extract_archive(mount)
            if extracted:
                return extracted
    raise FileNotFoundError("Dataset V2 was not found. Set DATASET_ROOT_OVERRIDE if necessary.")

DATASET_ROOT = find_dataset_root()
VAL_FOLDER = "valid" if (DATASET_ROOT / "images" / "valid").exists() else "val"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).resolve() if OUTPUT_ROOT_OVERRIDE else (
    Path("/kaggle/working/ppe_baseline_study") if Path("/kaggle/working").exists()
    else Path.cwd() / "ai-model" / "outputs" / "ppe_baseline_study"
)
RUNS_DIR = OUTPUT_ROOT / "runs"
RESULTS_DIR = OUTPUT_ROOT / "results"
CONFIG_DIR = OUTPUT_ROOT / "config"
RECOVERY_DIR = OUTPUT_ROOT / "recovery"
EXPORT_DIR = OUTPUT_ROOT / "export"
for folder in [RUNS_DIR, RESULTS_DIR, CONFIG_DIR, RECOVERY_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

data_config = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": f"images/{VAL_FOLDER}",
    "test": "images/test",
    "nc": 4,
    "names": CLASS_NAMES,
}
DATA_YAML = CONFIG_DIR / "dataset_v2.yaml"
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print("Dataset:", DATASET_ROOT)
print("Outputs:", OUTPUT_ROOT)

## 2. Dataset verification

This audit checks that all labels follow YOLO format and reports the image and bounding-box counts used by the baseline. It does not alter the dataset.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
rows = []
for split, folder in {"train": "train", "val": VAL_FOLDER, "test": "test"}.items():
    image_dir = DATASET_ROOT / "images" / folder
    label_dir = DATASET_ROOT / "labels" / folder
    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    for image_path in images:
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            raise FileNotFoundError(f"Missing label: {label_path}")
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f"Invalid YOLO label at {label_path}:{line_number}")
            class_id = int(float(values[0]))
            coordinates = list(map(float, values[1:]))
            if class_id not in CLASS_NAMES or not all(0 <= value <= 1 for value in coordinates):
                raise ValueError(f"Invalid annotation at {label_path}:{line_number}")
            rows.append({"split": split, "class_name": CLASS_NAMES[class_id]})

labels_df = pd.DataFrame(rows)
class_table = labels_df.groupby(["class_name", "split"]).size().unstack(fill_value=0).reindex(CLASS_NAMES.values())
image_table = pd.Series({
    split: len([p for p in (DATASET_ROOT / "images" / folder).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
    for split, folder in {"train": "train", "val": VAL_FOLDER, "test": "test"}.items()
}, name="images")
display(image_table.to_frame())
display(class_table)
class_table.plot.bar(figsize=(9, 4), color=["#1f4e79", "#70ad47", "#ed7d31"])
plt.title("Dataset V2 class distribution")
plt.xlabel("")
plt.ylabel("Bounding boxes")
plt.tight_layout()
plt.show()
class_table.to_csv(RESULTS_DIR / "class_distribution.csv")

## 3. Train the baseline

The training recipe is fixed before viewing test results. Moderate geometric and color augmentation is used, while the model architecture remains unchanged. If `last.pt` exists, the notebook resumes instead of restarting. On Kaggle T4 ×2, Distributed Data Parallel uses a global batch of 32, with 16 images on each GPU and `nbs=64` as the nominal effective-batch target.

In [ ]:
RUN_NAME = "E0_baseline_seed42_full"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

def completed_epochs():
    if not RESULTS_CSV.exists():
        return 0
    return len(pd.read_csv(RESULTS_CSV))

def stage_resume_files(source_last):
    source_last = Path(source_last).expanduser().resolve()
    if not source_last.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {source_last}")
    LAST_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    if source_last != LAST_WEIGHTS.resolve():
        shutil.copy2(source_last, LAST_WEIGHTS)

    best_candidates = []
    if PREVIOUS_BEST_WEIGHTS:
        best_candidates.append(Path(PREVIOUS_BEST_WEIGHTS).expanduser())
    best_candidates.append(source_last.with_name("best.pt"))
    previous_best = next((path for path in best_candidates if path.exists()), None)
    if previous_best:
        if previous_best.resolve() != BEST_WEIGHTS.resolve():
            shutil.copy2(previous_best, BEST_WEIGHTS)
        print("Preserved previous strongest checkpoint:", previous_best)
    else:
        print("Warning: previous best.pt was not found. Add the full weights folder or set PREVIOUS_BEST_WEIGHTS.")

    history_candidates = [source_last.parent / "results.csv", source_last.parent.parent / "results.csv"]
    previous_history = next((path for path in history_candidates if path.exists()), None)
    if previous_history and previous_history.resolve() != RESULTS_CSV.resolve():
        shutil.copy2(previous_history, RESULTS_CSV)
    return LAST_WEIGHTS

def save_recovery(trainer):
    RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
    for source in [Path(trainer.last), Path(trainer.best), Path(trainer.csv), Path(trainer.save_dir) / "args.yaml"]:
        if source.exists():
            shutil.copy2(source, RECOVERY_DIR / source.name)
    state = {"experiment": "E0_baseline", "completed_epochs": int(trainer.epoch) + 1, "updated_utc": datetime.now(timezone.utc).isoformat()}
    (RECOVERY_DIR / "state.json").write_text(json.dumps(state, indent=2), encoding="utf-8")

train_args = {
    "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH,
    "workers": WORKERS, "device": TRAIN_DEVICE, "seed": SEED, "deterministic": True,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.01, "cos_lr": True,
    "warmup_epochs": 3, "weight_decay": 0.0005, "hsv_h": 0.015,
    "hsv_s": 0.55, "hsv_v": 0.35, "degrees": 5, "translate": 0.10,
    "scale": 0.45, "fliplr": 0.5, "mosaic": 0.5, "mixup": 0.05,
    "close_mosaic": 10, "amp": True, "nbs": 64, "project": str(RUNS_DIR), "name": RUN_NAME,
    "exist_ok": True, "save": True, "save_period": SAVE_PERIOD, "val": True,
    "patience": 100, "plots": True,
}
manifest = {
    "experiment": "E0_baseline", "architecture": "yolo26n.pt", "seed": SEED,
    "classes": CLASS_NAMES, "dataset": str(DATASET_ROOT), "training": train_args,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__},
}
(RESULTS_DIR / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

if RUN_TRAINING:
    resume_path = Path(RESUME_WEIGHTS).expanduser() if RESUME_WEIGHTS else LAST_WEIGHTS
    if completed_epochs() >= EPOCHS and BEST_WEIGHTS.exists():
        print(f"Baseline already complete: {completed_epochs()}/{EPOCHS} epochs")
    elif resume_path.exists():
        resume_path = stage_resume_files(resume_path)
        print("Resuming baseline from staged checkpoint:", resume_path)
        model = YOLO(str(resume_path))
        model.add_callback("on_model_save", save_recovery)
        model.train(resume=True, data=str(DATA_YAML), device=TRAIN_DEVICE, workers=WORKERS, batch=BATCH, save_period=SAVE_PERIOD, plots=True)
    else:
        model = YOLO("yolo26n.pt")
        model.add_callback("on_model_save", save_recovery)
        model.train(**train_args)
else:
    print("Training disabled.")

## 4. Validation and learning curves

The main result is validation `mAP50-95`. Per-class recall is retained because missing the two violation classes (`no-helmet` and `no-vest`) is operationally important.

In [ ]:
ACTIVE_BEST_WEIGHTS = BEST_WEIGHTS if BEST_WEIGHTS.exists() else RECOVERY_DIR / "best.pt"
ACTIVE_RESULTS_CSV = RESULTS_CSV if RESULTS_CSV.exists() else RECOVERY_DIR / "results.csv"
if not ACTIVE_BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Baseline best.pt was not found in the run or recovery folder.")

metrics = YOLO(str(ACTIVE_BEST_WEIGHTS)).val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=1, device=EVAL_DEVICE, plots=True)
box = metrics.box
record = {
    "experiment": "E0_baseline", "map50_95": float(box.map), "map50": float(box.map50),
    "precision_macro": float(box.mp), "recall_macro": float(box.mr),
}
class_ids = [int(value) for value in np.asarray(box.ap_class_index).tolist()]
for index, class_id in enumerate(class_ids):
    record[f"{CLASS_NAMES[class_id]}_ap50_95"] = float(np.asarray(box.ap)[index])
    record[f"{CLASS_NAMES[class_id]}_recall"] = float(np.asarray(box.r)[index])
baseline_metrics = pd.DataFrame([record])
baseline_metrics.to_csv(RESULTS_DIR / "baseline_validation.csv", index=False)
display(baseline_metrics.T.rename(columns={0: "value"}))

history = pd.read_csv(ACTIVE_RESULTS_CSV)
history.columns = history.columns.str.strip()
metric_column = "metrics/mAP50-95(B)"
best_row = history.loc[history[metric_column].idxmax()]
print("Best training epoch:", int(best_row["epoch"]) + 1)
print("Highest recorded validation mAP50-95:", round(float(best_row[metric_column]), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=["train/box_loss", "val/box_loss"], ax=axes[0], title="Localization loss")
history.plot(x="epoch", y=["metrics/mAP50(B)", metric_column], ax=axes[1], title="Validation accuracy")
plt.tight_layout()
plt.show()

## 5. Baseline artifact

`best.pt`, `baseline_validation.csv`, `results.csv`, and `experiment_manifest.json` form the baseline record. Keep these files unchanged so later architecture experiments can be compared against the same reference.

In [ ]:
exported = EXPORT_DIR / "ppe_v2_yolo26n_baseline_best.pt"
shutil.copy2(ACTIVE_BEST_WEIGHTS, exported)
print("Baseline weights:", exported)
print("Validation metrics:", RESULTS_DIR / "baseline_validation.csv")
print("Training history:", ACTIVE_RESULTS_CSV)